In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict, deque
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/atp_2000_2024_model_ready.csv", low_memory=False)
df["tourney_date"] = pd.to_datetime(df["tourney_date"])
df = df.sort_values(["tourney_date", "match_num"]).reset_index(drop=True)

is_A = df["winner_is_A"].values.astype(bool)   # used to build A-vs-B features
print(df.shape)

(71463, 58)


In [2]:
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut], df.iloc[cut:]

baseline_acc = accuracy_score(test["target"], (test["rank_diff"] < 0).astype(int))
rank_model = LogisticRegression(max_iter=1000).fit(train[["rank_diff"]], train["target"])
rank_acc = accuracy_score(test["target"], rank_model.predict(test[["rank_diff"]]))
print("baseline:", round(baseline_acc, 3), " rank model:", round(rank_acc, 3))

baseline: 0.637  rank model: 0.636


In [3]:
matches_played = defaultdict(int)
exp_w = np.zeros(len(df)); exp_l = np.zeros(len(df))

for i, row in enumerate(df.itertuples(index=False)):
    w, l = row.winner_id, row.loser_id
    exp_w[i] = matches_played[w]    # READ before
    exp_l[i] = matches_played[l]
    matches_played[w] += 1         # UPDATE after
    matches_played[l] += 1

df["exp_diff"] = np.where(is_A, exp_w - exp_l, exp_l - exp_w)

In [4]:
train, test = df.iloc[:cut], df.iloc[cut:]
m = LogisticRegression(max_iter=1000).fit(train[["exp_diff"]], train["target"])
print("experience only:", round(accuracy_score(test["target"], m.predict(test[["exp_diff"]])), 3))

experience only: 0.571


In [5]:
elo = defaultdict(lambda: 1500.0)
games = defaultdict(int)
elo_w = np.zeros(len(df)); elo_l = np.zeros(len(df))

def dynamic_k(n):
    return 250 / ((n + 5) ** 0.4)

for i, row in enumerate(df.itertuples(index=False)):
    w, l = row.winner_id, row.loser_id
    a, b = elo[w], elo[l]
    elo_w[i] = a; elo_l[i] = b                       # READ first
    expected_w = 1 / (1 + 10 ** ((b - a) / 400))
    elo[w] = a + dynamic_k(games[w]) * (1 - expected_w)
    elo[l] = b - dynamic_k(games[l]) * (1 - expected_w)
    games[w] += 1; games[l] += 1                     # UPDATE after

df["elo_diff"] = np.where(is_A, elo_w - elo_l, elo_l - elo_w)

In [6]:
names = df.drop_duplicates("winner_id").set_index("winner_id")["winner_name"]
top = pd.Series(elo).sort_values(ascending=False).head(5)
for pid, rating in top.items():
    print(names.get(pid, pid), round(rating))

Novak Djokovic 2265
Jannik Sinner 2262
Roger Federer 2190
Carlos Alcaraz 2177
Daniil Medvedev 2152


In [7]:
train, test = df.iloc[:cut], df.iloc[cut:]
elo_rule = accuracy_score(test["target"], (test["elo_diff"] > 0).astype(int))
elo_model = LogisticRegression(max_iter=1000).fit(train[["elo_diff"]], train["target"])
elo_acc = accuracy_score(test["target"], elo_model.predict(test[["elo_diff"]]))

print("baseline:  ", round(baseline_acc, 3))
print("Elo rule:  ", round(elo_rule, 3))
print("Elo model: ", round(elo_acc, 3))

baseline:   0.637
Elo rule:   0.642
Elo model:  0.642


In [8]:
surface_elo = defaultdict(lambda: 1500.0)
surface_games = defaultdict(int)
selo_w = np.zeros(len(df)); selo_l = np.zeros(len(df))

for i, row in enumerate(df.itertuples(index=False)):
    w, l, s = row.winner_id, row.loser_id, row.surface
    a, b = surface_elo[(w, s)], surface_elo[(l, s)]
    selo_w[i] = a; selo_l[i] = b                      # READ first
    exp = 1 / (1 + 10 ** ((b - a) / 400))
    surface_elo[(w, s)] = a + dynamic_k(surface_games[(w, s)]) * (1 - exp)
    surface_elo[(l, s)] = b - dynamic_k(surface_games[(l, s)]) * (1 - exp)
    surface_games[(w, s)] += 1; surface_games[(l, s)] += 1

df["surface_elo_diff"] = np.where(is_A, selo_w - selo_l, selo_l - selo_w)

In [9]:
names = df.drop_duplicates("winner_id").set_index("winner_id")["winner_name"]
for surf in ["Clay", "Grass", "Hard"]:
    ratings = {p: v for (p, s), v in surface_elo.items()
               if s == surf and surface_games[(p, surf)] >= 50}
    top = pd.Series(ratings).sort_values(ascending=False).head(3)
    print(surf, "->", [f"{names.get(p, p)} {round(v)}" for p, v in top.items()])

Clay -> ['Rafael Nadal 2255', 'Novak Djokovic 2150', 'Stefanos Tsitsipas 2100']
Grass -> ['Novak Djokovic 2211', 'Roger Federer 2102', 'Rafael Nadal 1970']
Hard -> ['Jannik Sinner 2282', 'Novak Djokovic 2264', 'Roger Federer 2200']


In [10]:
recent = defaultdict(lambda: deque(maxlen=10))
form_w = np.full(len(df), 0.5); form_l = np.full(len(df), 0.5)

for i, row in enumerate(df.itertuples(index=False)):
    w, l = row.winner_id, row.loser_id
    if recent[w]: form_w[i] = sum(recent[w]) / len(recent[w])   # READ first
    if recent[l]: form_l[i] = sum(recent[l]) / len(recent[l])
    recent[w].append(1); recent[l].append(0)                    # UPDATE after

df["form_diff"] = np.where(is_A, form_w - form_l, form_l - form_w)

In [11]:
train, test = df.iloc[:cut], df.iloc[cut:]
def acc(cols):
    m = LogisticRegression(max_iter=1000).fit(train[cols], train["target"])
    return round(accuracy_score(test["target"], m.predict(test[cols])), 4)

print("baseline:             ", round(baseline_acc, 4))
print("Elo:                  ", acc(["elo_diff"]))
print("Elo + surface:        ", acc(["elo_diff", "surface_elo_diff"]))
print("Elo + surface + form: ", acc(["elo_diff", "surface_elo_diff", "form_diff"]))

baseline:              0.6366
Elo:                   0.6425
Elo + surface:         0.6461
Elo + surface + form:  0.6446


In [12]:
h2h = defaultdict(int)
h2h_w = np.zeros(len(df))
for i, row in enumerate(df.itertuples(index=False)):
    w, l = row.winner_id, row.loser_id
    h2h_w[i] = h2h[(w, l)] - h2h[(l, w)]     # READ prior edge (0 if never met)
    h2h[(w, l)] += 1                          # UPDATE
df["h2h_diff"] = np.where(is_A, h2h_w, -h2h_w)

In [13]:
r = df[df["winner_name"].isin(["Novak Djokovic","Rafael Nadal"]) &
       df["loser_name"].isin(["Novak Djokovic","Rafael Nadal"])]
print("Djokovic:", (r["winner_name"]=="Novak Djokovic").sum(),
      " Nadal:", (r["winner_name"]=="Rafael Nadal").sum())

Djokovic: 30  Nadal: 29


In [14]:
last_date = {}
rest_w = np.zeros(len(df)); rest_l = np.zeros(len(df))
for i, row in enumerate(df.itertuples(index=False)):
    w, l, d = row.winner_id, row.loser_id, row.tourney_date
    rest_w[i] = (d - last_date[w]).days if w in last_date else 30
    rest_l[i] = (d - last_date[l]).days if l in last_date else 30
    last_date[w] = d; last_date[l] = d
df["rest_diff"] = np.where(is_A, rest_w - rest_l, rest_l - rest_w)

In [15]:
rest_w_cap = np.minimum(rest_w, 60); rest_l_cap = np.minimum(rest_l, 60)
df["rest_cap_diff"] = np.where(is_A, rest_w_cap - rest_l_cap, rest_l_cap - rest_w_cap)

train, test = df.iloc[:cut], df.iloc[cut:]
def acc(cols):
    m = LogisticRegression(max_iter=1000).fit(train[cols], train["target"])
    return round(accuracy_score(test["target"], m.predict(test[cols])), 4)
base = ["elo_diff", "surface_elo_diff"]
print("raw:   ", acc(base + ["rest_diff"]))
print("capped:", acc(base + ["rest_cap_diff"]))

raw:    0.6479
capped: 0.6465


In [16]:
base = ["elo_diff", "surface_elo_diff"]
print("baseline:      ", round(baseline_acc, 4))
print("Elo + surface: ", acc(base))
print("+ h2h:         ", acc(base + ["h2h_diff"]))
print("+ rest:        ", acc(base + ["rest_diff"]))
print("+ experience:  ", acc(base + ["exp_diff"]))
print("+ all three:   ", acc(base + ["h2h_diff", "rest_diff", "exp_diff"]))

baseline:       0.6366
Elo + surface:  0.6461
+ h2h:          0.6472
+ rest:         0.6479
+ experience:   0.6468
+ all three:    0.6495
